# Парсинг IMDB аггрегатора

Иморт всех необходимых библиотек:

In [ ]:
import requests
from bs4 import BeautifulSoup
import re
from datetime import datetime
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd
from selenium.webdriver.support.ui import WebDriverWait
import time

## Получение страницы через request

Это первый способ получить код страницы. Он имеет свои плюсы и минусы в контексте данного ресурса

- **Плюсы**
  - Быстро. Нет траты времени на отработку скриптов в браузере.
- **Минусы**
  - imdb не отдает страницы без header и без cookie. Что требует определенных манипуляций с сессией.
  - Код страницы до отработки скриптов.

## Получение страницы через selenium 

Это второй способ. 

- **Плюсы**
  - Код страницы после отработки скриптов.
  - Клиентская сессия с куками, не требуется особых манипуляцый.
- **Минусы**
  - Медленно. Требуется время на отработку скриптов.
  - Присутствие капчи. Каждые 30 минут imdb запрашивает капчу, что ломает процесс сбора данных.


## Задание необходимых функций

Request главной страницы фильма. В headers должны быть куки незалогиненной сессии. Их можно взять из браузера в хедерах запроса главной страницы.

In [ ]:
def get_film_page(film_id: str) :
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/120.0.0.0 Safari/537.36",
        "Accept-Language": "en-US,en;q=0.9",
        "Cookie": "COOKIE_WITHOUT_LOGIN"
    }

    
    site = requests.get(f'https://www.imdb.com/title/{film_id}', headers=headers)  

    return site.text

Request страницы оценок фильма. В headers должны быть куки залогиненной сессии. Их можно взять из браузера в хедерах запроса главной страницы.

In [ ]:
def get_review_page(film_id: str, rate: int) :

    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/120.0.0.0 Safari/537.36",
        "Accept-Language": "en-US,en;q=0.9",
        "Cookie": "COOKIE_WITH_LOGIN"
    }

    site = requests.get(f'https://www.imdb.com/title/{film_id}/reviews/?rating={rate}&ref_=tt_ururv_histo_{rate}', headers=headers)  

    return site.text

Функции конвертации строк, времени в нужные типы форматы данных

In [ ]:
def time_to_minute(s: str) :
    # Hh Mm -> u16
    try: 
        match = re.fullmatch(r"(?:(\d+)h)?\s*(?:(\d+)m)?", s.strip())
        minutes = int(match.group(1) or 0) * 60 + int(match.group(2) or 0)
        return minutes
    except :
        return None

def date_to(s: str) :
    try: 
        date = datetime.strptime(s.split(" (")[0], "%B %d, %Y")
        result = date.strftime("%Y-%m-%d")

        return result
    except :
        return None

def money_to_int(s: str) :
    try:
        value = int(re.sub(r"\D", "", s))

        return value
    except :
        return None

def short_int_to_canonical(s: str) -> int:
    try:
        s = s.strip().upper().replace(",", "")
        match = re.fullmatch(r"(\d+(?:\.\d+)?)\s*([KM]?)", s)

        if not match:
            raise ValueError(f"Invalid number: {s}")

        number = float(match.group(1))
        multiplier = {"": 1, "K": 1_000, "M": 1_000_000}[match.group(2)]

        result = int(number * multiplier)

        if not 0 <= result <= 2**64 - 1:
            raise OverflowError("Value exceeds u64 range")

        return result
    except :
        return None

Функции парсинга страниц

In [ ]:
def get_sub_region(soup, tag, obj) :
    try :
        return soup.find_all(tag, obj)[0]
    except :
        return None

def get_sub_text(soup, tag, obj) :
    try :
        return soup.find_all(tag, obj)[0].text
    except :
        return None

def imdb_parse(page: str) :
    soup = BeautifulSoup(page)

    user_reviews_section = get_sub_region(soup, 'section', {'cel_widget_id': 'StaticFeature_UserReviews'})
    users_reviews = get_sub_text(user_reviews_section, 'span', {'class': 'ipc-title__subtext'})
    users_rate = get_sub_text(user_reviews_section, 'span', {'class': 'ipc-rating-star--rating'})
    users_scores = get_sub_text(user_reviews_section, 'span', {'data-testid': 'rating-histogram-vote-count'})

    common_section = get_sub_region(soup, 'div', {'class': 'sc-65e7d1cb-6 eOoLYF'})
    counts = common_section.find_all('li', {'class': 'ipc-inline-list__item sc-cc1bacb6-1 lmpOrh'}) if common_section != None else []


    critics_reviews = None
    metascore = None

    for i in counts :
        if "Critic" in i.text :
            critics_reviews = i.find_all('span', {'class': 'score'})[0].text
        if "Metascore" in i.text :
            metascore = i.find_all('span', {'class': 'score'})[0].text

    watchers = get_sub_text(soup, 'div', {'class': 'sc-aef509ad-2 kUKnTZ'}).replace('Added by ', '').replace(' users', '') if get_sub_text(soup, 'div', {'class': 'sc-aef509ad-2 kUKnTZ'}) != None else None
    popularity = get_sub_text(soup, 'div', {'data-testid': 'hero-rating-bar__popularity__score'})

    details_section = get_sub_region(soup, 'section', {'cel_widget_id': 'StaticFeature_Details'})
    release = get_sub_region(details_section, 'li', {'data-testid': "title-details-releasedate"})
    release_time = get_sub_text(release, 'a', {'class': 'ipc-metadata-list-item__list-content-item ipc-metadata-list-item__list-content-item--link'})
    origin = get_sub_region(details_section, 'li', {'data-testid': "title-details-origin"})
    origin_country = get_sub_text(origin, 'li', {'class': 'ipc-inline-list__item'})
    lang = get_sub_region(details_section, 'li', {'data-testid': "title-details-languages"})
    lang_code = get_sub_text(lang, 'a', {'class': 'ipc-metadata-list-item__list-content-item ipc-metadata-list-item__list-content-item--link'})
    filming = get_sub_region(details_section, 'li', {'data-testid': "title-details-filminglocations"})
    filming_loc = get_sub_text(filming, 'a', {'class': 'ipc-metadata-list-item__list-content-item ipc-metadata-list-item__list-content-item--link'})
    company = get_sub_region(details_section, 'li', {'data-testid': "title-details-companies"})
    company_main = get_sub_text(company, 'a', {'class': 'ipc-metadata-list-item__list-content-item ipc-metadata-list-item__list-content-item--link'})

    storyline_section = get_sub_region(soup, 'section', {'cel_widget_id': 'StaticFeature_Storyline'})
    storyline = get_sub_region(storyline_section, 'li', {'data-testid': "storyline-genres"})
    genre = get_sub_text(storyline, 'a', {'class': 'ipc-metadata-list-item__list-content-item ipc-metadata-list-item__list-content-item--link'})
    
    budget_section = get_sub_region(soup, 'section', {'cel_widget_id': 'StaticFeature_BoxOffice'})
    budget = get_sub_region(budget_section, 'li', {'data-testid': "title-boxoffice-budget"})
    budget_rev = get_sub_text(budget, 'span', {'class': 'ipc-metadata-list-item__list-content-item ipc-btn--not-interactable'})
    grossdomestic = get_sub_region(budget_section, 'li', {'data-testid': "title-boxoffice-grossdomestic"})
    grossdomestic_rev = get_sub_text(grossdomestic, 'span', {'class': 'ipc-metadata-list-item__list-content-item ipc-btn--not-interactable'})
    cumulativeworldwidegross = get_sub_region(budget_section, 'li', {'data-testid': "title-boxoffice-cumulativeworldwidegross"})
    cumulativeworldwidegross_rev = get_sub_text(cumulativeworldwidegross, 'span', {'class': 'ipc-metadata-list-item__list-content-item ipc-btn--not-interactable'})
    openingweekenddomestic = get_sub_region(budget_section, 'li', {'data-testid': "title-boxoffice-openingweekenddomestic"})
    openingweekenddomestic_rev = get_sub_text(openingweekenddomestic, 'span', {'class': 'ipc-metadata-list-item__list-content-item ipc-btn--not-interactable'})

    tech_section = get_sub_region(soup, 'section', {'cel_widget_id': 'StaticFeature_TechSpecs'})
    runtime = get_sub_region(tech_section, 'li', {'data-testid': "title-techspec_runtime"})
    runtime = get_sub_text(runtime, 'span', {'class': 'ipc-metadata-list-item__list-content-item ipc-btn--not-interactable'})
    color = get_sub_region(tech_section, 'li', {'data-testid': "title-techspec_color"})
    color = get_sub_text(color, 'a', {'class': 'ipc-metadata-list-item__list-content-item ipc-metadata-list-item__list-content-item--link'})
    aspectratio = get_sub_region(tech_section, 'li', {'data-testid': "title-techspec_aspectratio"})
    aspectratio = get_sub_text(aspectratio, 'span', {'class': 'ipc-metadata-list-item__list-content-item ipc-btn--not-interactable'})

    main_section = get_sub_region(soup, 'div', {'class': 'sc-6339114b-0 ckcdKn'})
    title = get_sub_text(main_section, 'span', {'data-testid': "hero__primary-text"})
    lis = main_section.find_all('li', {'class': "ipc-inline-list__item"})
    certificate = None

    for i in lis :
        if "tt_ov_pg" in str(i) :
            certificate = get_sub_text(i, 'a', {'class': 'ipc-link ipc-link--baseAlt ipc-link--inherit-color'})



    return [title, certificate, time_to_minute(runtime), genre, color, aspectratio, date_to(release_time), origin_country, lang_code, filming_loc, company_main, money_to_int(budget_rev), money_to_int(grossdomestic_rev), money_to_int(cumulativeworldwidegross_rev), money_to_int(openingweekenddomestic_rev), short_int_to_canonical(watchers), money_to_int(popularity), short_int_to_canonical(critics_reviews), int(metascore) if metascore != None else None, short_int_to_canonical(users_reviews), short_int_to_canonical(users_scores), float(users_rate) if users_rate != None else None]

def imdb_rates_parse(page: str) :
    soup = BeautifulSoup(page)
    title = soup.find_all('div', {'data-testid': "tturv-total-reviews"})[0].text
    return short_int_to_canonical(title.replace(' reviews', '').replace(' review', ''))

def get_film_count(page: str) :
    soup = BeautifulSoup(page)
    matches = re.findall(r'All\s+([\d,]+)', page)

    numbers = [int(x.replace(",", "")) for x in matches]
    return numbers[0] if len(numbers) > 0 else 0


## Обработка категории

In [ ]:
br = webdriver.Chrome()


2 тестовые ячейки, чтобы потестить парсинг, работу со страницей в selenium

In [ ]:
tt_id = 'tt8814476'
br.get(f'https://www.imdb.com/title/{tt_id}')
br.execute_script("window.scrollTo(0, document.body.scrollHeight/2)")
br.implicitly_wait(10)
print("URL:", br.current_url)
print("Ready:", br.execute_script("return document.readyState"))
selector = 'section[cel_widget_id="StaticFeature_Storyline"]'

for i in range(30):
    result = br.execute_script("""
        return document.querySelector(
            'section[cel_widget_id="StaticFeature_Storyline"]'
        )?.outerHTML || null;
    """)

    if result:
        print("FOUND at", i)
        break

    time.sleep(1)
else:
    print("NOT FOUND")

print(br.execute_script("""
    return [...document.querySelectorAll('section')]
        .map(x => x.getAttribute('cel_widget_id'))
        .filter(x => x && x.includes('Storyline'));
"""))
main_page = br.page_source
row = imdb_parse(main_page)
print(row)

In [ ]:
tt_id = 'tt8814476'
br.get(f'https://www.imdb.com/title/{tt_id}')
br.implicitly_wait(10)
br.execute_script("window.scrollTo(0, document.body.scrollHeight)")
br.implicitly_wait(10)

time.sleep(30)
print(br.execute_script("return document.readyState"))

main_page = br.page_source
print("StaticFeature_Storyline" in main_page)
print("Storyline" in main_page)
row = imdb_parse(main_page)
print(row)

Обределяем жанр

In [ ]:
genre = 'fantasy'
out_file = 'data/films_imdb'
add_film_limit = 300

Подготавливаем список фильмов по данному жанру. При первом прогоне попросит ввести капчу, после ввода требуется запустить ячейку еще раз.

In [ ]:
try:
    data = pd.read_csv(f'{out_file}.csv', sep=';')
except:
    data = pd.DataFrame([], columns=["title", "certificate", "duration", "genre", "color", "aspectratio", "release_time", "origin_country", "lang_code", "filming_loc", "company_main", "budget_rev", "grossdomestic_rev", "cumulativeworldwidegross_rev", "openingweekenddomestic_rev", "watchers", "popularity", "critics_reviews", "metascore", "users_reviews", "users_scores", "users_rate", "imdb_id", "score_1", "score_2", "score_3", "score_4", "score_5", "score_6", "score_7", "score_8", "score_9", "score_10"])

In [ ]:
try:
    data = pd.read_csv(f'{out_file}.csv', sep=';')
except:
    data = pd.DataFrame([], columns=["title", "certificate", "duration", "genre", "color", "aspectratio", "release_time", "origin_country", "lang_code", "filming_loc", "company_main", "budget_rev", "grossdomestic_rev", "cumulativeworldwidegross_rev", "openingweekenddomestic_rev", "watchers", "popularity", "critics_reviews", "metascore", "users_reviews", "users_scores", "users_rate", "imdb_id", "score_1", "score_2", "score_3", "score_4", "score_5", "score_6", "score_7", "score_8", "score_9", "score_10"])
br.get(f'https://www.imdb.com/interest/{genre}/movies-shows')
main_page = br.page_source
film_count = get_film_count(main_page)-100
print(film_count)
fields = br.find_elements(By.CSS_SELECTOR, ".MoviesAndShowsTab_grid__FeR5g")
links = []
for i in range(len(fields)) :
    fields = br.find_elements(By.CSS_SELECTOR, ".MoviesAndShowsTab_grid__FeR5g")
    elements = fields[i].find_elements(By.CSS_SELECTOR, ".MoviesAndShowsTab_gridCell__9_GSc")
    if i > 0 :
        max_len = len(elements)
        while len(elements) < film_count and max_len <= 4000:
            br.execute_script("window.scrollTo(0, document.body.scrollHeight)")
            br.implicitly_wait(5)
            time.sleep(1)
            elements = fields[i].find_elements(By.CSS_SELECTOR, ".MoviesAndShowsTab_gridCell__9_GSc")
            max_len = len(elements)
            print(f'mf: {max_len}')
    fields = br.find_elements(By.CSS_SELECTOR, ".MoviesAndShowsTab_grid__FeR5g")
    elements = fields[i].find_elements(By.CSS_SELECTOR, ".MoviesAndShowsTab_gridCell__9_GSc")
    print(len(elements))
    def get_elem_data (x):
        html_str = x.get_attribute("outerHTML")
        soup = BeautifulSoup(html_str)
        title = soup.find_all('span', {'data-testid': 'title'})[0].text
        url = soup.find_all('a', {'class': 'ipc-lockup-overlay ipc-focusable ipc-focusable--constrained'})[0].get("href")
        return (title, url)
    pull_data = list(map(get_elem_data, elements))
    print(len(pull_data))
    links.extend(pull_data)
    br.execute_script("window.scrollTo(0, document.body.scrollHeight)")
    br.implicitly_wait(5)
    br.execute_script("window.scrollTo(0, document.body.scrollHeight)")
    br.implicitly_wait(5)
    br.execute_script("window.scrollTo(0, document.body.scrollHeight)")
    br.implicitly_wait(5)
    time.sleep(2)
print(len(links))



In [ ]:
dataset = data[["title", "certificate", "duration", "genre", "color", "aspectratio", "release_time", "origin_country", "lang_code", "filming_loc", "company_main", "budget_rev", "grossdomestic_rev", "cumulativeworldwidegross_rev", "openingweekenddomestic_rev", "watchers", "popularity", "critics_reviews", "metascore", "users_reviews", "users_scores", "users_rate", "imdb_id", "score_1", "score_2", "score_3", "score_4", "score_5", "score_6", "score_7", "score_8", "score_9", "score_10"]].values.tolist()
i = 0+303

Обработка главных страниц фильма, требуется быть залогиненым. Требуется следить за временем с момента ввода капчи(30 минут), остановить обработку требуется до, после ручного обновления страницы с вводом капчи, запустить ячейку снова.

In [ ]:

new_df = pd.DataFrame(dataset, columns=["title", "certificate", "duration", "genre", "color", "aspectratio", "release_time", "origin_country", "lang_code", "filming_loc", "company_main", "budget_rev", "grossdomestic_rev", "cumulativeworldwidegross_rev", "openingweekenddomestic_rev", "watchers", "popularity", "critics_reviews", "metascore", "users_reviews", "users_scores", "users_rate", "imdb_id", "score_1", "score_2", "score_3", "score_4", "score_5", "score_6", "score_7", "score_8", "score_9", "score_10"])
print(new_df.shape, len(links))
for url in links :
    tt_id = re.search(r"tt\d+", url[1]).group()
    if i >= add_film_limit + 100 :
        break
    if tt_id in new_df['imdb_id'].values :
        continue
    try:
        br.get(f'https://www.imdb.com/title/{tt_id}')
        br.execute_script("window.scrollTo(0, document.body.scrollHeight/2)")
        selector = 'section[cel_widget_id="StaticFeature_Storyline"]'

        element = None
        muls = [0.5,0.25,0.75,0.4,0.6,0.85]
        for j in muls:
            elements = br.find_elements(By.CSS_SELECTOR, selector)

            if elements:
                element = elements[0]
                break

            br.execute_script(f'window.scrollTo(0, document.body.scrollHeight*{j})')
            time.sleep(0.1)

        if element:
            print(element.get_attribute("outerHTML"))
        else:
            print("Not found")
        br.implicitly_wait(5)
        time.sleep(1)
        main_page = br.page_source
        row = imdb_parse(main_page)
        row.append(tt_id)
        try:
            for rate in range(1, 11) :
                br.get(f'https://www.imdb.com/title/{tt_id}/reviews/?rating={rate}&ref_=tt_ururv_histo_{rate}')
                main_page = br.page_source
                try:
                    row.append(imdb_rates_parse(main_page))
                except :
                    row.append(None)
        except :
            break
        print(row)
    except:
        break
    dataset.append(row)
    i += 1
    br.implicitly_wait(5)
    try:
        time.sleep(2)
    except:
        break
new_df = pd.DataFrame(dataset, columns=["title", "certificate", "duration", "genre", "color", "aspectratio", "release_time", "origin_country", "lang_code", "filming_loc", "company_main", "budget_rev", "grossdomestic_rev", "cumulativeworldwidegross_rev", "openingweekenddomestic_rev", "watchers", "popularity", "critics_reviews", "metascore", "users_reviews", "users_scores", "users_rate", "imdb_id", "score_1", "score_2", "score_3", "score_4", "score_5", "score_6", "score_7", "score_8", "score_9", "score_10"])
print((i, new_df.shape))


In [ ]:

new_df.to_csv(f'{out_file}.csv', sep=';', index=False)
new_df.head()

In [ ]:
result = new_df.loc[new_df["genre"].isna(), "imdb_id"].tolist()
result

'tt26545566', 'tt43619249', 'tt33407001', 'tt40231601', tt20454624, tt12117810, tt9684936, tt12844100, tt41087748, tt29282591, tt31809887

In [ ]:
dataset = data.values.tolist()
i = 0
dataset

Обработка фильмов с помощью requests

In [ ]:
req_df = pd.DataFrame(dataset, columns=["title", "certificate", "duration", "genre", "color", "aspectratio", "release_time", "origin_country", "lang_code", "filming_loc", "company_main", "budget_rev", "grossdomestic_rev", "cumulativeworldwidegross_rev", "openingweekenddomestic_rev", "watchers", "popularity", "critics_reviews", "metascore", "users_reviews", "users_scores", "users_rate", "imdb_id", "score_1", "score_2", "score_3", "score_4", "score_5", "score_6", "score_7", "score_8", "score_9", "score_10"])

for url in links :
    tt_id = re.search(r"tt\d+", url[1]).group()
    if i >= add_film_limit + 100 :
        break
    if tt_id in data['imdb_id'].values :
        continue
    try:
        main_page = get_film_page(tt_id)

        row = imdb_parse(main_page)

        for r in range(1, 11) :
            row.append(imdb_rates_parse(get_review_page(tt_id, r)))
    except:
        break
    dataset.append(row)
    br.implicitly_wait(5)
    time.sleep(2)
    i += 1
req_df = pd.DataFrame(dataset, columns=["title", "certificate", "duration", "genre", "color", "aspectratio", "release_time", "origin_country", "lang_code", "filming_loc", "company_main", "budget_rev", "grossdomestic_rev", "cumulativeworldwidegross_rev", "openingweekenddomestic_rev", "watchers", "popularity", "critics_reviews", "metascore", "users_reviews", "users_scores", "users_rate", "imdb_id", "score_1", "score_2", "score_3", "score_4", "score_5", "score_6", "score_7", "score_8", "score_9", "score_10"])
req_df.head()